# TCC — Sentinel-2 em toda a RGI de Guaxupé

Habilite Internet e GPU T4 no Kaggle. Anexe o backup completo U-Net/U-Mamba como entrada privada. Rode as células em ordem.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone

CODE_REF = "dab29621eb3b769ddcaf7956945459f697caedc4"
WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "tcc_nova_cena"
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/oguel/tcc-umamba.git", str(REPO)], check=True)
if subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip():
    raise RuntimeError("Preserve as alterações no checkout antes de continuar.")
subprocess.run(["git", "fetch", "origin", CODE_REF], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", CODE_REF], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == CODE_REF
print("Código fixado:", CODE_REF)


## Localizar os backups corretos

A célula verifica os hashes dos melhores pesos. Se o backup estiver compactado, aceita somente o ZIP completo já auditado e extrai para uma pasta nova de trabalho. Não modifica as entradas ou os checkpoints originais.

In [ ]:
EXPECTED = {
    "unet": "addd3b5f680c1495dfa73b7eae6332fe085015c8d3e7035d825dfab4adc1d2cf",
    "umamba": "038a100adc397ae7482bf12550f958c9046829d5ff3d4f16d56275f69a91f9bf",
}
def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

roots = [WORK / "tcc_runs", INPUT, WORK / "backup_final_extraido"]
def locate():
    runs = {}
    for kind, expected in EXPECTED.items():
        candidates = [p for root in roots if root.exists() for p in root.rglob(f"best_{kind}_rgi.pt")]
        for path in sorted(candidates):
            if sha256(path) == expected:
                runs[kind] = path.parent
                break
    return runs

RUNS = locate()
if len(RUNS) != 2:
    candidates = [p for p in INPUT.rglob("*.zip") if "backup" in p.name.lower() and p.stat().st_size == 53710616]
    complete = [p for p in candidates if sha256(p) == "234cb97b21931b4c3b9ec8140003b17ca5319a03cfa899b297140160a6b61a1a"]
    if complete:
        destination = WORK / "backup_final_extraido"
        if destination.exists():
            raise RuntimeError("Extração anterior incompleta: preserve a pasta e revise o problema.")
        with zipfile.ZipFile(complete[0]) as archive:
            entries = archive.infolist()
            if len(entries) > 500 or sum(e.file_size for e in entries) > 512 * 1024**2:
                raise RuntimeError("Dimensão do arquivo inesperada.")
            for entry in entries:
                resolved = (destination / entry.filename).resolve()
                if not resolved.is_relative_to(destination.resolve()):
                    raise RuntimeError("Caminho inválido no ZIP.")
            if archive.testzip() is not None:
                raise RuntimeError("ZIP corrompido.")
            destination.mkdir()
            archive.extractall(destination)
        RUNS = locate()
if len(RUNS) != 2:
    raise RuntimeError("Anexe como entrada privada o backup_COMPLETO_UNET_UMAMBA.zip ou as duas pastas de execução completas.")
print({kind: str(path) for kind, path in RUNS.items()})


## Preparar o ambiente CUDA

A instalação e a inferência ocorrem em processos separados.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-runtime.txt"), "geopandas"], check=True)
subprocess.run([sys.executable, "-c", "from src.models.umamba_runtime import install_prebuilt_colab_stack; print('PyTorch alterado:', install_prebuilt_colab_stack())"], cwd=REPO, check=True)


## Compor o mosaico regional e gerar os mapas

O ZIP inclui o mosaico de previsões, a cobertura dos nove municípios e a origem por pixel. A busca usa a malha regional 310044 do IBGE e prioriza datas com mais pixels válidos.

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output = WORK / f"sentinel2_rgi_{stamp}"
subprocess.run([
    sys.executable, "-u", "-m", "scripts.infer_new_sentinel",
    "--unet-run", str(RUNS["unet"]),
    "--umamba-run", str(RUNS["umamba"]),
    "--output", str(output),
], cwd=REPO, check=True)
archive = shutil.make_archive(str(output), "zip", root_dir=output)
print("BAIXAR E PRESERVAR:", archive)
print("SHA-256:", sha256(Path(archive)))
from IPython.display import Image, display
display(Image(filename=str(output / "comparacao.png")))
